# Exponential Boltzmann energy distribution

## Prerequisites

Review probability-density normalization, the Boltzmann factor, and numerical integration.

## Objectives

1. Represent the thermal energy $	heta=k_B T$ with explicit physical units.
2. Evaluate the normalized exponential density
   $$f(E)=	heta^{-1}\exp(-E/	heta),\qquad E\geq 0.$$
3. Check its finite-interval probability against the analytical integral.
4. Plot and interpret distributions at several thermal-energy scales.

This density assumes a constant density of states. It is **not** the three-dimensional translational kinetic-energy distribution, which includes a $\sqrt{E}$ density-of-states factor.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.statmech.distributions.boltzmann_energy import (
    BoltzmannExponentialEnergyDistribution,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

In [ ]:
minimum_energy_electron_volts = 0.0
maximum_energy_electron_volts = 1.0
energy_sample_count = 4001
thermal_energy_values_electron_volts = (0.025, 0.05, 0.1)
energy_unit = PhysicalUnit(expression="electron_volt")
energy_values_electron_volts = np.linspace(
    minimum_energy_electron_volts,
    maximum_energy_electron_volts,
    energy_sample_count,
    dtype=np.float64,
)
energies = VectorQuantity(
    magnitude=energy_values_electron_volts,
    unit=energy_unit,
)
distributions = tuple(
    BoltzmannExponentialEnergyDistribution(
        thermal_energy=ScalarQuantity(
            magnitude=thermal_energy_electron_volts,
            unit=energy_unit,
        )
    )
    for thermal_energy_electron_volts in thermal_energy_values_electron_volts
)
evaluations = tuple(
    distribution.evaluate(energies=energies) for distribution in distributions
)

In [ ]:
integration_relative_tolerance = 1.0e-5
integration_absolute_tolerance = 2.0e-8
for evaluation in evaluations:
    thermal_energy = evaluation.request.distribution.thermal_energy.magnitude
    represented_probability = np.trapezoid(
        evaluation.probability_density.magnitude,
        evaluation.request.energies.magnitude,
    )
    expected_probability = 1.0 - np.exp(-maximum_energy_electron_volts / thermal_energy)
    assert np.isclose(
        represented_probability,
        expected_probability,
        rtol=integration_relative_tolerance,
        atol=integration_absolute_tolerance,
    )
    assert np.all(evaluation.probability_density.magnitude >= 0.0)

In [ ]:
figure_size_inches = (8.0, 4.8)
figure, axis = plt.subplots(figsize=figure_size_inches)
for evaluation in evaluations:
    thermal_energy = evaluation.request.distribution.thermal_energy.magnitude
    axis.plot(
        evaluation.request.energies.magnitude,
        evaluation.probability_density.magnitude,
        label=rf"$	heta={thermal_energy:.3f}$ eV",
    )
axis.set(
    xlabel="energy (eV)",
    ylabel="probability density (1/eV)",
    title="Exponential Boltzmann energy distributions",
)
axis.grid(True)
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

Each curve has its largest density at zero energy and decays on the scale $	heta$. Increasing $	heta$ broadens the distribution and lowers its value at the origin so that the total probability remains one on $[0,\infty)$.

The finite plotting interval captures $1-\exp(-E_{\max}/	heta)$ rather than exactly one. The numerical check compares against that finite-interval probability.

## Exercises

1. Derive the cumulative distribution function and median energy.
2. Increase $E_{\max}$ and quantify the omitted tail probability.
3. Convert the energy coordinates to joules and verify that the integrated probability is unchanged.
4. Compare this constant-density-of-states model with the three-dimensional translational kinetic-energy distribution.
